In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import optuna
from tqdm.auto import tqdm
import warnings, subprocess, os, gc

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Environment Detection ─────────────────────────────────────────────────────
IS_KAGGLE = os.path.exists("/kaggle/input")

def has_local_gpu():
    try:
        result = subprocess.run(["nvidia-smi"], capture_output=True, timeout=5)
        return result.returncode == 0
    except Exception:
        return False

USE_GPU  = IS_KAGGLE or has_local_gpu()
DATA_DIR = "/kaggle/input/playground-series-s6e3/" if IS_KAGGLE else "../data/"
SUB_DIR  = "/kaggle/working/"                       if IS_KAGGLE else "../"

print(f"Environment  : {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"GPU          : {'Enabled ✓' if USE_GPU else 'Disabled — CPU'}")
print(f"Data dir     : {DATA_DIR}")
print(f"Output dir   : {SUB_DIR}")
print(f"XGBoost ver  : {xgb.__version__}")

# ── Settings ──────────────────────────────────────────────────────────────────
RUN_TUNING   = True
N_TRIALS     = 50
N_CV_FOLDS   = 5
SEEDS        = [42, 2024, 777, 1337, 999]
N_SPLITS     = 10
TOTAL_MODELS = len(SEEDS) * N_SPLITS
TARGET       = "Churn"

# Smoothing factor for target encoding (higher = more regularisation)
TE_ALPHA = 10

PRECOMPUTED_PARAMS = {
    "learning_rate":      0.02,
    "max_depth":          6,
    "max_leaves":         50,
    "min_child_weight":   5,
    "subsample":          0.8,
    "colsample_bytree":   0.8,
    "colsample_bylevel":  0.8,
    "reg_alpha":          0.1,
    "reg_lambda":         1.0,
    "gamma":              0.05,
}

In [ ]:
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

print(f"Train : {train.shape}  |  Test : {test.shape}")

# Target
train[TARGET] = (train[TARGET] == "Yes").astype(int)
churn_rate    = train[TARGET].mean()
scale_pos_w   = round((1 - churn_rate) / churn_rate, 4)
print(f"Churn rate       : {churn_rate:.4f}")
print(f"scale_pos_weight : {scale_pos_w}")

sub = test[["id"]].copy()

# ── Joint Preprocessing ───────────────────────────────────────────────────────
combined = pd.concat([train.drop(TARGET, axis=1), test], axis=0).reset_index(drop=True)
n_train  = len(train)

num_cols = combined.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in combined.select_dtypes(include="object").columns if c != "id"]

# pandas 3.0-safe fillna
for c in num_cols:
    combined[c] = combined[c].fillna(combined[c].median())
for c in cat_cols:
    combined[c] = combined[c].fillna("Missing")

print(f"Numeric cols     ({len(num_cols)}): {num_cols}")
print(f"Categorical cols ({len(cat_cols)}): {cat_cols}")

In [ ]:
def add_base_fe(df):
    """Existing FE from nb10 — the proven 8 features."""
    d = df.copy()
    num_base = ["tenure", "MonthlyCharges", "TotalCharges"]
    d["num_sum"]  = d[num_base].sum(axis=1)
    d["num_mean"] = d[num_base].mean(axis=1)
    d["num_std"]  = d[num_base].std(axis=1)
    d["num_max"]  = d[num_base].max(axis=1)
    d["num_min"]  = d[num_base].min(axis=1)
    d["Average_Monthly_Actual"] = d["TotalCharges"] / (d["tenure"] + 1e-5)
    d["Monthly_diff"]           = d["MonthlyCharges"] - d["Average_Monthly_Actual"]
    d["Monthly_ratio"]          = d["MonthlyCharges"] / (d["Average_Monthly_Actual"] + 1e-5)
    return d

combined = add_base_fe(combined)
print("Base FE applied. New columns:", ["num_sum","num_mean","num_std","num_max","num_min",
                                         "Average_Monthly_Actual","Monthly_diff","Monthly_ratio"])

In [ ]:
class TargetEncoder:
    """
    Smoothed target encoder.

    Smoothing formula (Micci-Barreca 2001):
        encoded = (n_cat * mean_cat + alpha * mean_global) / (n_cat + alpha)

    alpha: smoothing strength.
        High alpha (e.g. 50) → pulls rare categories toward global mean (safe).
        Low  alpha (e.g.  5) → trusts category mean more (risky on small groups).
    """

    def __init__(self, cols, alpha=10):
        self.cols        = cols
        self.alpha       = alpha
        self.global_mean = None
        self.maps        = {}

    def fit(self, X, y):
        self.global_mean = y.mean()
        for col in self.cols:
            stats = (
                pd.concat([X[[col]], y.rename("target")], axis=1)
                .groupby(col)["target"]
                .agg(["sum", "count"])
            )
            stats["encoded"] = (
                (stats["sum"] + self.alpha * self.global_mean)
                / (stats["count"] + self.alpha)
            )
            self.maps[col] = stats["encoded"].to_dict()
        return self

    def transform(self, X):
        Xt = X.copy()
        for col in self.cols:
            Xt[col + "_te"] = (
                X[col].map(self.maps[col]).fillna(self.global_mean)
            )
        return Xt

    def fit_transform(self, X, y):
        return self.fit(X, y).transform(X)

# Columns to target-encode
TE_COLS = cat_cols.copy()   # all 15 categorical cols

print(f"Target-encoding {len(TE_COLS)} columns with alpha={TE_ALPHA}:")
for c in TE_COLS:
    n_unique = combined[c].nunique()
    print(f"  {c:25s}  ({n_unique} unique values)")

In [ ]:
# ── Label encode categorical cols (kept for tree splits) ─────────────────────
le = LabelEncoder()
for c in cat_cols:
    combined[c] = le.fit_transform(combined[c].astype(str))

# Split back for global TE
train_fe = combined.iloc[:n_train].copy().reset_index(drop=True)
test_fe  = combined.iloc[n_train:].copy().reset_index(drop=True)
y_train  = train[TARGET].reset_index(drop=True)

# ── Global Target Encoding (used in Optuna tuning — minor leakage acceptable) ─
global_te = TargetEncoder(cols=TE_COLS, alpha=TE_ALPHA)
train_fe  = global_te.fit_transform(train_fe, y_train)
test_fe   = global_te.transform(test_fe)

# ── Feature List ──────────────────────────────────────────────────────────────
TE_NEW_COLS = [c + "_te" for c in TE_COLS]
FEATURES    = [c for c in train_fe.columns if c not in ["id", TARGET]]

X      = train_fe[FEATURES]
y      = y_train
X_test = test_fe[FEATURES]

print(f"X shape        : {X.shape}")
print(f"X_test shape   : {X_test.shape}")
print(f"Total features : {len(FEATURES)}")
print(f"  - Original + FE cols : {len(FEATURES) - len(TE_NEW_COLS)}")
print(f"  - Target-encoded cols: {len(TE_NEW_COLS)} (suffix _te)")
print(f"\nNew _te columns:")
for c in TE_NEW_COLS:
    print(f"  {c}")

In [ ]:
DEVICE = "cuda" if USE_GPU else "cpu"

base_params = {
    "objective":        "binary:logistic",
    "eval_metric":      "auc",
    "tree_method":      "hist",
    "device":           DEVICE,
    "grow_policy":      "lossguide",
    "scale_pos_weight": scale_pos_w,
    "seed":             42,
    "verbosity":        0,
    "nthread":          -1,
}
print(f"Device: {DEVICE}  |  grow_policy: lossguide  |  scale_pos_weight: {scale_pos_w}")

dtrain_full = xgb.DMatrix(X, label=y)

if RUN_TUNING:
    print(f"\nStarting Optuna: {N_TRIALS} trials × {N_CV_FOLDS}-fold CV")
    pbar        = tqdm(total=N_TRIALS, desc="Optuna Trials", unit="trial")
    best_so_far = [0.0]

    def objective(trial):
        params = {
            **base_params,
            "learning_rate":     trial.suggest_float("learning_rate",     0.005, 0.1,  log=True),
            "max_depth":         trial.suggest_int(  "max_depth",         3,     9),
            "max_leaves":        trial.suggest_int(  "max_leaves",        15,    127),
            "min_child_weight":  trial.suggest_float("min_child_weight",  1,     20),
            "subsample":         trial.suggest_float("subsample",         0.5,   1.0),
            "colsample_bytree":  trial.suggest_float("colsample_bytree",  0.4,   1.0),
            "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.4,   1.0),
            "reg_alpha":         trial.suggest_float("reg_alpha",         1e-8,  10.0, log=True),
            "reg_lambda":        trial.suggest_float("reg_lambda",        1e-8,  10.0, log=True),
            "gamma":             trial.suggest_float("gamma",             1e-8,  5.0,  log=True),
        }
        cv        = xgb.cv(
            params, dtrain_full,
            num_boost_round       = 2000,
            nfold                 = N_CV_FOLDS,
            stratified            = True,
            early_stopping_rounds = 50,
            seed                  = 42,
            verbose_eval          = False,
        )
        score     = cv["test-auc-mean"].max()
        best_iter = int(cv["test-auc-mean"].idxmax())
        trial.set_user_attr("best_iteration", best_iter)

        if score > best_so_far[0]:
            best_so_far[0] = score
        pbar.set_postfix({"AUC": f"{score:.5f}", "Best": f"{best_so_far[0]:.5f}"})
        pbar.update(1)
        return score

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=N_TRIALS)
    pbar.close()

    best_trial  = study.best_trial
    best_iter   = best_trial.user_attrs.get("best_iteration", 500)
    best_params = {**base_params, **best_trial.params}

    print(f"\n{'='*55}")
    print(f"Best CV AUC    : {best_trial.value:.5f}")
    print(f"Best iteration : {best_iter}")
    for k, v in best_trial.params.items():
        print(f"  {k:25s}: {v}")
    print(f"{'='*55}")

    study.trials_dataframe().to_csv(
        os.path.join(SUB_DIR, "xgb_te_tuning_results.csv"), index=False
    )
else:
    print("Using PRECOMPUTED_PARAMS.")
    best_params = {**base_params, **PRECOMPUTED_PARAMS}
    best_iter   = 500

In [ ]:
print(f"Multi-Seed Ensemble: {len(SEEDS)} seeds × {N_SPLITS} folds = {TOTAL_MODELS} models")
print(f"OOF target encoding applied INSIDE each fold (no leakage)\n")

# Restore un-encoded versions for fold-wise TE
# We need raw label-encoded cats (before global TE was applied)
# Re-derive from combined which still has LE-encoded cat cols
train_raw = combined.iloc[:n_train].copy().reset_index(drop=True)
test_raw  = combined.iloc[n_train:].copy().reset_index(drop=True)
FEATURES_RAW = [c for c in train_raw.columns if c not in ["id", TARGET]]

X_raw      = train_raw[FEATURES_RAW]
X_test_raw = test_raw[FEATURES_RAW]

test_preds_sum = np.zeros(len(X_test_raw))
global_oof_sum = np.zeros(len(X_raw))
fold_auc_log   = []

outer_bar = tqdm(SEEDS, desc="Seeds", position=0)

for seed in outer_bar:
    outer_bar.set_description(f"Seed {seed}")
    skf      = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    seed_oof = np.zeros(len(X_raw))

    inner_bar = tqdm(
        enumerate(skf.split(X_raw, y)),
        total    = N_SPLITS,
        desc     = "  Folds",
        position = 1,
        leave    = False,
    )

    for fold, (tr_idx, val_idx) in inner_bar:
        X_tr_raw, X_val_raw = X_raw.iloc[tr_idx],  X_raw.iloc[val_idx]
        y_tr,     y_val     = y.iloc[tr_idx],       y.iloc[val_idx]

        # ── OOF Target Encoding: fit on train fold, apply to val + test ──────
        fold_te = TargetEncoder(cols=TE_COLS, alpha=TE_ALPHA)
        fold_te.fit(X_tr_raw, y_tr)

        X_tr_enc  = fold_te.transform(X_tr_raw)
        X_val_enc = fold_te.transform(X_val_raw)
        X_tst_enc = fold_te.transform(X_test_raw)

        # Feature list after TE
        feat_cols = [c for c in X_tr_enc.columns if c != "id"]

        dtrain = xgb.DMatrix(X_tr_enc[feat_cols],  label=y_tr)
        dval   = xgb.DMatrix(X_val_enc[feat_cols],  label=y_val)
        dtest  = xgb.DMatrix(X_tst_enc[feat_cols])

        fold_params = {**best_params, "seed": seed}
        model = xgb.train(
            fold_params,
            dtrain,
            num_boost_round       = best_iter + 200,
            evals                 = [(dval, "val")],
            early_stopping_rounds = 100,
            verbose_eval          = False,
        )

        val_preds  = model.predict(dval,  iteration_range=(0, model.best_iteration))
        test_preds = model.predict(dtest, iteration_range=(0, model.best_iteration))

        fold_auc = roc_auc_score(y_val, val_preds)
        fold_auc_log.append(fold_auc)

        seed_oof[val_idx]        = val_preds
        global_oof_sum[val_idx] += val_preds
        test_preds_sum          += test_preds

        inner_bar.set_postfix({"fold_auc": f"{fold_auc:.5f}", "best_iter": model.best_iteration})

        del model, dtrain, dval, dtest, fold_te
        gc.collect()

    seed_auc = roc_auc_score(y, seed_oof)
    outer_bar.set_postfix({"seed_oof_auc": f"{seed_auc:.5f}"})

global_oof = global_oof_sum / len(SEEDS)
final_auc  = roc_auc_score(y, global_oof)

print(f"\n{'='*55}")
print(f"Fold AUC — mean  : {np.mean(fold_auc_log):.5f}  std: {np.std(fold_auc_log):.5f}")
print(f"Global OOF AUC   : {final_auc:.5f}")
print(f"{'='*55}")

In [ ]:
final_preds = test_preds_sum / TOTAL_MODELS
sub[TARGET] = final_preds

out_file = os.path.join(SUB_DIR, "submission_xgb_target_enc.csv")
sub.to_csv(out_file, index=False)

print(f"Saved  → {out_file}")
print(f"Range  : [{final_preds.min():.4f}, {final_preds.max():.4f}]")
print(f"Mean   : {final_preds.mean():.4f}")
display(sub.head())

In [ ]:
result = subprocess.run([
    "kaggle", "competitions", "submit",
    "-c", "playground-series-s6e3",
    "-f", os.path.join(SUB_DIR, "submission_xgb_target_enc.csv"),
    "-m", "XGB scaledpos+lossguide OOF target-encoding 15 cats alpha=10 50-model ensemble",
], capture_output=True, text=True)

print(result.stdout)
if result.stderr.strip():
    print("STDERR:", result.stderr)